In [1]:
# Clean reinstall of numpy and gensim
!pip uninstall -y numpy gensim
!pip install numpy==1.24.4  # Known to work well with gensim
!pip install gensim

Found existing installation: numpy 1.24.4
Uninstalling numpy-1.24.4:
  Successfully uninstalled numpy-1.24.4
  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.24.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.24.4 which is incompatible.
blosc2 3.3.2 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.24.4 which is incompatible.
pymc 5.22.0 requires numpy>=1.25.0, but you have numpy 1.24.4 which is

  Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (26.7 MB)


In [6]:
# LDA

import string
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer


# Creating example documents
doc_1 = "A whopping 96.5 percent of water on Earth is in our oceans, covering 71 percent of the surface of our planet. And at any given time, about 0.001 percent is floating above us in the atmosphere. If all of that water fell as rain at once, the whole planet would get about 1 inch of rain."

doc_2 = "One-third of your life is spent sleeping. Sleeping 7-9 hours each night should help your body heal itself, activate the immune system, and give your heart a break. Beyond that--sleep experts are still trying to learn more about what happens once we fall asleep."

doc_3 = "A newborn baby is 78 percent water. Adults are 55-60 percent water. Water is involved in just about everything our body does."

doc_4 = "While still in high school, a student went 264.4 hours without sleep, for which he won first place in the 10th Annual Great San Diego Science Fair in 1964."

doc_5 = "We experience water in all three states: solid ice, liquid water, and gas water vapor."

# Create corpus
corpus = [doc_1, doc_2, doc_3, doc_4, doc_5]

####################################

# remove stopwords, punctuation, and normalize the corpus
stop = set(stopwords.words('english'))
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()

def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    punc_free = "".join(ch for ch in stop_free if ch not in exclude)
    normalized = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    return normalized

clean_corpus = [clean(doc).split() for doc in corpus]

for c in clean_corpus:
    print(c)

################################################
from gensim import corpora

# Creating document-term matrix
dictionary = corpora.Dictionary(clean_corpus)
doc_term_matrix = [dictionary.doc2bow(doc) for doc in clean_corpus]

['whopping', '965', 'percent', 'water', 'earth', 'ocean', 'covering', '71', 'percent', 'surface', 'planet', 'given', 'time', '0001', 'percent', 'floating', 'u', 'atmosphere', 'water', 'fell', 'rain', 'once', 'whole', 'planet', 'would', 'get', '1', 'inch', 'rain']
['onethird', 'life', 'spent', 'sleeping', 'sleeping', '79', 'hour', 'night', 'help', 'body', 'heal', 'itself', 'activate', 'immune', 'system', 'give', 'heart', 'break', 'beyond', 'thatsleep', 'expert', 'still', 'trying', 'learn', 'happens', 'fall', 'asleep']
['newborn', 'baby', '78', 'percent', 'water', 'adult', '5560', 'percent', 'water', 'water', 'involved', 'everything', 'body', 'doe']
['still', 'high', 'school', 'student', 'went', '2644', 'hour', 'without', 'sleep', 'first', 'place', '10th', 'annual', 'great', 'san', 'diego', 'science', 'fair', '1964']
['experience', 'water', 'three', 'state', 'solid', 'ice', 'liquid', 'water', 'gas', 'water', 'vapor']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# OPTIONAL: Following would save the dictionary and doc_term_matrix and then load them back later.

# Save dictionary
dictionary.save("dictionary.dict")

# Save doc_term_matrix (corpus)
import pickle
with open("corpus.pkl", "wb") as f:
    pickle.dump(doc_term_matrix, f)


# Load dictionary
from gensim import corpora
dictionary = corpora.Dictionary.load("dictionary.dict")

# Load corpus
with open("corpus.pkl", "rb") as f:
    doc_term_matrix = pickle.load(f)

In [7]:
############################################

# Latent Semantic Analysis (LSA) (also known as Latent Semantic Indexing, or LSI)
from gensim.models import LsiModel


# This line creates the LSA model. Here's what each parameter means:
# doc_term_matrix: A document-term matrix (usually a list of bag-of-words vectors for each document).
# num_topics=3: The number of latent topics you want to extract from the corpus.
# id2word=dictionary: A Gensim dictionary mapping word IDs to actual words, created from your text corpus.
lsa = LsiModel(doc_term_matrix, num_topics=3, id2word = dictionary)

# LSA model
print("LSA:")
print(lsa.print_topics(num_topics=3, num_words=3))

# OPTIONAL
# lsa.save("lsa_model.gensim")
# lsa_loaded = LsiModel.load("lsa_model.gensim")

LSA:
[(0, '0.555*"water" + 0.489*"percent" + 0.239*"planet"'), (1, '0.361*"sleeping" + 0.215*"hour" + 0.215*"still"'), (2, '-0.562*"water" + 0.231*"rain" + 0.231*"planet"')]


In [3]:
# Latent Dirichlet Allocation (LDA)
from gensim.models import LdaModel

# Create the LDA model
# doc_term_matrix: Bag-of-words representation of documents.
# num_topics=3: The number of topics you want to extract.
# id2word=dictionary: Word-ID mapping.
# passes=10: The number of times the model will loop over the corpus to improve results.
lda = LdaModel(doc_term_matrix, num_topics=3, id2word=dictionary, passes=10, random_state=42)

# Print the topics
print("LDA:")
print(lda.print_topics(num_topics=3, num_words=3))

# save and load the model
# lda.save("lda_model.gensim")
# lda_loaded = LdaModel.load("lda_model.gensim")

LDA:
[(0, '0.032*"hour" + 0.032*"sleeping" + 0.032*"still"'), (1, '0.012*"still" + 0.012*"percent" + 0.012*"sleeping"'), (2, '0.102*"water" + 0.065*"percent" + 0.029*"planet"')]
